In [11]:
import os

from dotenv import load_dotenv

load_dotenv()
from google.genai import types

In [12]:
GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY")

In [13]:
CHITCHAT_MODEL = "gemini-2.0-flash"
CHITCHAT_INSTRUCTIONS = "You are an agent to answer questions about the construction progress.You greet user and do basic cit chat.( greetings, how are you, etc.).Do not answer any out of scope questions."


In [14]:
print(CHITCHAT_INSTRUCTIONS)

You are an agent to answer questions about the construction progress.You greet user and do basic cit chat.( greetings, how are you, etc.).Do not answer any out of scope questions.


In [15]:
CHITCHAT_CONFIG = types.GenerateContentConfig(
    temperature=0.7,
    max_output_tokens=200
)

In [16]:
from google.adk.agents import LlmAgent

chitchat_agent = LlmAgent(
    name="chitchat_agent",
    model=CHITCHAT_MODEL,
    description="A simple agent that chit chats with the user.",
    instruction= CHITCHAT_INSTRUCTIONS,
    generate_content_config=CHITCHAT_CONFIG,
    

    
)

## Setup Session Service and Runner

In [17]:
import os
import asyncio
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner


# --- Session Management ---
# Key Concept: SessionService stores conversation history & state.
# InMemorySessionService is simple, non-persistent storage for this tutorial.
session_service = InMemorySessionService()

# Define constants for identifying the interaction context
USER_ID = "user_123"
SESSION_ID = "chitchat_session"
APP_NAME = "chit_chat_app"

# Create the specific session where the conversation will happen
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

# --- Runner ---
# Key Concept: Runner orchestrates the agent execution loop.
runner = Runner(
    agent=chitchat_agent, # The agent we want to run
    app_name=APP_NAME,   # Associates runs with our app
    session_service=session_service # Uses our session manager
)
print(f"Runner created for agent '{runner.agent.name}'.")

Session created: App='chit_chat_app', User='user_123', Session='chitchat_session'
Runner created for agent 'chitchat_agent'.


In [21]:
# @title Define Agent Interaction Function


async def call_agent_async(query: str, runner, user_id, session_id):
  """Sends a query to the agent and prints the final response."""
  print(f"\n>>> User Query: {query}")

  # Prepare the user's message in ADK format
  content = types.Content(role='user', parts=[types.Part(text=query)])

  final_response_text = "Agent did not produce a final response." # Default

  # Key Concept: run_async executes the agent logic and yields Events.
  # We iterate through events to find the final answer.
  async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
      # You can uncomment the line below to see *all* events during execution
      # print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}, Content: {event.content}")

      # Key Concept: is_final_response() marks the concluding message for the turn.
      if event.is_final_response():
          if event.content and event.content.parts:
             # Assuming text response in the first part
             final_response_text = event.content.parts[0].text
          elif event.actions and event.actions.escalate: # Handle potential errors/escalations
             final_response_text = f"Agent escalated: {event.error_message or 'No specific message.'}"
          # Add more checks here if needed (e.g., specific error codes)
          break # Stop processing events once the final response is found

  print(f"<<< Agent Response: {final_response_text}")


In [19]:
# await call_agent_async(
#     "Hello, how are you?",
#     runner,
#     USER_ID,
#     SESSION_ID
# )

In [ ]:
# @title Run the Initial Conversation

# We need an async function to await our interaction helper
async def run_conversation():
    await call_agent_async("Hi",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID)

    await call_agent_async("What can you do ? ",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID) # Expecting the tool's error message

    await call_agent_async("How is the weather today ?",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID)

# Execute the conversation using await in an async context (like Colab/Jupyter)
await run_conversation()




>>> User Query: Hi
<<< Agent Response: Hello there! How are you doing today?


>>> User Query: What can you do ? 
<<< Agent Response: I can provide updates on the construction progress. Just let me know what you're interested in!


>>> User Query: How is the weather today ?
<<< Agent Response: I am sorry, I am not able to provide information that is not related to construction progress.

